### IMPORT ALL THE TOOLS UPFRONT


In [21]:
import os
from dotenv import load_dotenv

from langchain_community.document_loaders import TextLoader
#### langchain_community is a package, documnet_loader is a class, textloader is a method

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import JinaEmbeddings

In [2]:
load_dotenv()

True

In [4]:
groq_key = os.getenv("GROQ_API_KEY")
jina_key = os.getenv("JINA_API_KEY")


In [6]:
DATA_FILE_PATH = os.path.join("data", "hr_policy.txt")

#### DATA INGESTION

In [8]:
loader = TextLoader(DATA_FILE_PATH , encoding="utf-8")
documents = loader.load()

print(documents)

### loader is object of the method textloader

[Document(metadata={'source': 'data\\hr_policy.txt'}, page_content="IMPORTANT NOTICE\nThis document is synthetic content created for testing and development of an HR policy retrieval and question-answering project. It does not represent the actual policies of MAYA or any real organization.\n\n1. PURPOSE AND SCOPE\nThis fictional handbook provides policies for employees, interns, associates, directors, managers, and HR administrators. It covers attendance, leave, remote work, conduct, performance, confidentiality, information security, reimbursements, onboarding, internships, communication, grievances, and separation.\n\n2. WORKING HOURS AND ATTENDANCE\n2.1 Standard Working Hours\nThe standard schedule is Monday to Friday, 9:30 AM to 6:30 PM. Core collaboration hours are 10:00 AM to 5:00 PM.\n\n2.2 Attendance\nTeam members are expected to attend scheduled meetings, complete assigned work within agreed timelines, and communicate proactively when unavailable.\n\n2.3 Late Arrival\nIf a tea

In [10]:

print(f"Total number of characters in the document: {len(documents[0].page_content)}")

Total number of characters in the document: 18639


#### SPLIT OUR DATA

In [16]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

chunks = text_splitter.split_documents(documents)
print(chunks)
len(chunks)

[Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='IMPORTANT NOTICE\nThis document is synthetic content created for testing and development of an HR policy retrieval and question-answering project. It does not represent the actual policies of MAYA or any real organization.'), Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='1. PURPOSE AND SCOPE\nThis fictional handbook provides policies for employees, interns, associates, directors, managers, and HR administrators. It covers attendance, leave, remote work, conduct, performance, confidentiality, information security, reimbursements, onboarding, internships, communication, grievances, and separation.'), Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='2. WORKING HOURS AND ATTENDANCE\n2.1 Standard Working Hours\nThe standard schedule is Monday to Friday, 9:30 AM to 6:30 PM. Core collaboration hours are 10:00 AM to 5:00 PM.\n\n2.2 Attendance\nTeam members are expected to attend scheduled

44

In [20]:
print(chunks[41].page_content)

16. GRIEVANCE AND CONCERNS
Team members may raise workplace concerns with their reporting manager, HR representative, or designated organizational contact. Concerns should be handled sensitively and shared only with people who need the information to address the matter. Retaliation against a person who raises a concern in good faith is not permitted.


### EMBEDD THE DATA

In [27]:
embeddings_model = JinaEmbeddings(model_name="jina-embeddings-v2-base-en")
print(" my embedding model is ready and the name is:", embeddings_model.model_name)

 my embedding model is ready and the name is: jina-embeddings-v2-base-en


### STORE DATA IN VECTOR DB

In [28]:
from langchain_community.vectorstores import FAISS 

vector_store = FAISS.from_documents(chunks, embeddings_model)

print("CHUNKS ARE STORED", vector_store.index.ntotal)

CHUNKS ARE STORED 44


In [29]:
test_query = "How many sick leaves employees get"
## SIMILARITY SEARCH

top_matches = vector_store.similarity_search(test_query, k=2)
print(f"Query: {test_query}\n")
for i,match in enumerate(top_matches,start=1):
    print(f"--- Match {i} ---")
    print(match.page_content)
    print()


Query: How many sick leaves employees get

--- Match 1 ---
3.2 Sick Leave
Employees may take up to 12 days of sick leave per calendar year. Supporting documentation may be requested for extended absence.

3.3 Leave Requests
Planned leave should normally be submitted at least 3 working days in advance. Approval depends on team requirements and workload.

3.4 Emergency Leave
Emergency leave may be requested when advance notice is not reasonably possible. The employee should notify their manager as soon as possible.

--- Match 2 ---
3.2 Sick Leave
Employees may take up to 12 days of sick leave per calendar year. Supporting documentation may be requested for extended absence.

3.3 Leave Requests
Planned leave should normally be submitted at least 3 working days in advance. Approval depends on team requirements and workload.

3.4 Emergency Leave
Emergency leave may be requested when advance notice is not reasonably possible. The employee should notify their manager as soon as possible.



### DATA RETRIVAL PIPELINE

#### LLM

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model = "openai/gpt-oss-120b",
    temperature=0.9 ## creativity of the model
)

llm.model_name

'openai/gpt-oss-120b'

In [42]:
tr = llm.invoke(" Hey what is life and what is the purpose of life ? answer in one line amye in a joke manner ")
tr.content

'Life’s just a cosmic Wi‑Fi password—everyone’s trying to connect, but the purpose is just to keep the router (your brain) from overheating!'

### TOOL


In [51]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

def search_hr_policy(question:str)->str:
    """
    Search the HR policy documnet for information about leave, work from home, probation, notice period, reimbursement, code of conduct, holidays or exit process.

    """

    matching_chunks = retriever.invoke(question)
    return "\n\n".join(chunk.page_content for chunk in matching_chunks)

### AI AGENT

LLM - BRAIN
TOOL - SUPER POWER 
MEMORY - no memory

In [43]:
from langchain.agents import create_agent

In [52]:
hr_assistant = create_agent(
    model = llm,
    tools=[search_hr_policy],
    system_prompt="""

    You are a friendly HR assistent.
    Always use the search_hr_policy tool to look up 
    facts before answering. If the answer isn't in the search result, say you don't know
    instead of guessing.

    """
)

print("HR assistent agent is ready to answer the questions!")

HR assistent agent is ready to answer the questions!


In [64]:
response = hr_assistant.invoke(
    {
        "messages":[
            {
                "role":"user",
                "content":"Hey, which organisation work for?"
            }
        ]
    }
)

In [65]:
response["messages"][-1].content

'The handbook I’m referencing is a **synthetic, test‑only document** – it isn’t tied to a real‑world company. The text does mention the name **“MAYA,”** but it also explicitly states that the policies, numbers, dates, procedures, and organizational details are fictional and do **not represent the actual policies of MAYA or any real organization**. So, there isn’t a real organisation that I work for; the content is purely for testing purposes.'